# 후보 모델에 대해 하이퍼파라미터 선택
### 후보 모델
1. 로지스틱 회귀 + 특성 v7 사용

In [1]:
# 데이터셋 로드
import pandas as pd

train_origin = pd.read_csv("../data/processed/01/train.csv")

## 후보 1 테스트


In [2]:
from src.feature import prep_v7
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

prep = prep_v7()
model = make_pipeline(prep, LogisticRegression())

파라미터 조합 설정

In [3]:
param_grid = [{
    "logisticregression__C": [0.001, 0.01, 0.1,0.5, 1, 10, 100],
    "logisticregression__l1_ratio": [1],
    "logisticregression__max_iter": [10000],
    "logisticregression__solver": ["liblinear"],
     "logisticregression__class_weight": [None, "balanced"]
},
    {
    "logisticregression__C": [0.001, 0.01, 0.1,0.5, 1, 10, 100],
    "logisticregression__l1_ratio": [0],
    "logisticregression__max_iter": [10000],
    "logisticregression__solver": ["lbfgs"],
         "logisticregression__class_weight": [None, "balanced"]
},
]

하이퍼 파라미터 탐색 객체 생성

In [4]:
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)


학습 및 결과확인


In [5]:
train = train_origin.copy()
labels = train["Survived"]

grid_search.fit(train, labels)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'logisticregression__C': 1, 'logisticregression__class_weight': None, 'logisticregression__l1_ratio': 0, 'logisticregression__max_iter': 10000, 'logisticregression__solver': 'lbfgs'}
0.8244164286417808


In [6]:

results = grid_search.cv_results_
#딕셔너리 형태라 데이터프레임으로 바꿔보는게 좋음
results_df = pd.DataFrame(results).sort_values("rank_test_score")
results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_logisticregression__C,param_logisticregression__class_weight,param_logisticregression__l1_ratio,param_logisticregression__max_iter,param_logisticregression__solver,params,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
22,0.014953,0.002799,0.006423,0.001630,1.000,NaN,0,10000,lbfgs,"{'logisticregression__C': 1, 'logisticregressi...",...,0.824416,0.037120,1,0.838313,0.811951,0.819298,0.829825,0.828070,0.825491,0.009072
20,0.013136,0.001404,0.005558,0.000848,0.500,NaN,0,10000,lbfgs,"{'logisticregression__C': 0.5, 'logisticregres...",...,0.823008,0.035669,2,0.840070,0.810193,0.821053,0.826316,0.833333,0.826193,0.010257
24,0.014409,0.002825,0.005565,0.000677,10.000,NaN,0,10000,lbfgs,"{'logisticregression__C': 10, 'logisticregress...",...,0.821609,0.043335,3,0.841828,0.803163,0.819298,0.829825,0.826316,0.824086,0.012754
12,0.030096,0.010228,0.005652,0.001549,100.000,NaN,1,10000,liblinear,"{'logisticregression__C': 100, 'logisticregres...",...,0.820191,0.046662,4,0.841828,0.808436,0.817544,0.828070,0.826316,0.824439,0.011160
8,0.013480,0.000487,0.006020,0.001600,1.000,NaN,1,10000,liblinear,"{'logisticregression__C': 1, 'logisticregressi...",...,0.818792,0.040542,5,0.838313,0.810193,0.821053,0.828070,0.829825,0.825491,0.009419
26,0.015056,0.003062,0.006395,0.001115,100.000,NaN,0,10000,lbfgs,"{'logisticregression__C': 100, 'logisticregres...",...,0.818783,0.047057,6,0.841828,0.808436,0.817544,0.828070,0.826316,0.824439,0.011160
10,0.020147,0.002005,0.005610,0.000626,10.000,NaN,1,10000,liblinear,"{'logisticregression__C': 10, 'logisticregress...",...,0.817384,0.044600,7,0.841828,0.804921,0.817544,0.826316,0.828070,0.823736,0.012207
6,0.013469,0.002844,0.007247,0.003091,0.500,NaN,1,10000,liblinear,"{'logisticregression__C': 0.5, 'logisticregres...",...,0.815976,0.033747,8,0.840070,0.804921,0.824561,0.826316,0.828070,0.824788,0.011326
18,0.017616,0.005372,0.007376,0.005153,0.100,NaN,0,10000,lbfgs,"{'logisticregression__C': 0.1, 'logisticregres...",...,0.814587,0.033484,9,0.833040,0.804921,0.817544,0.819298,0.828070,0.820575,0.009669
9,0.015916,0.004089,0.006068,0.001242,1.000,balanced,1,10000,liblinear,"{'logisticregression__C': 1, 'logisticregressi...",...,0.813188,0.035114,10,0.831283,0.794376,0.807018,0.812281,0.817544,0.812500,0.012139


### 최적 모델 저장

In [7]:
import joblib
best_model = grid_search.best_estimator_
joblib.dump(best_model, "../data/model/tuned_logistic_regression.pkl")

['../data/model/tuned_logistic_regression.pkl']